# EXP-002 visible object heuristic explorer

Routed scoring notebook: click-only, keyboard-only, and keyboard-click policies. It adds connected-component target selection, no-op suppression, action-family routing, and EXP-001 fallback.

Validation gate before submission: local score >= 0.2124 and local levels >= 7/183.


In [ ]:
!python -m pip install -q --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv pyarrow


In [ ]:

import os, json, random, zlib, math
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import dotenv
import arc_agi
from arcengine import GameAction, GameState

EXP_ID = "EXP-002"
MAX_MOVES = 1000
SEED = 42
USE_PER_GAME_SEED = False

WORK = Path("/kaggle/working")
ART = WORK / "exp002_artifacts"
ART.mkdir(exist_ok=True)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    get_ipython().system("curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games")
    mode = "online"
    envdir = ""
else:
    mode = "offline"
    envdir = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/"

(WORK / ".env").write_text(f"""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE={mode}
ENVIRONMENTS_DIR={envdir}
RECORDINGS_DIR=/kaggle/working/server_recording
""")
dotenv.load_dotenv(WORK / ".env", override=True)

MOVE_DELTAS = {
    GameAction.ACTION1: (0, -1),  # up
    GameAction.ACTION2: (0,  1),  # down
    GameAction.ACTION3: (-1, 0),  # left
    GameAction.ACTION4: (1,  0),  # right
}
MOVE_ACTIONS = list(MOVE_DELTAS.keys())

def rng_for(game_id: str) -> random.Random:
    seed = SEED if not USE_PER_GAME_SEED else SEED + (zlib.adler32(game_id.encode()) & 0xFFFFFFFF)
    return random.Random(seed)

def frame_hash(frame) -> int | None:
    if frame is None:
        return None
    return zlib.adler32(np.asarray(frame, dtype=np.uint8).tobytes()) & 0xFFFFFFFF

def get_frame(obs):
    if obs is None or not getattr(obs, "frame", None):
        return None
    return np.asarray(obs.frame[-1])

def random_visible_pixel(frame, rng):
    colors = np.unique(frame).tolist()
    color = rng.choice(colors)
    ys, xs = np.where(frame == color)
    i = rng.randint(0, len(xs) - 1)
    return {"x": int(xs[i]), "y": int(ys[i])}

def connected_components(frame, min_area=2, max_area=500):
    """Palette-color connected components, ignoring dominant background/wall colors."""
    arr = np.asarray(frame)
    h, w = arr.shape[:2]
    vals, counts = np.unique(arr, return_counts=True)
    count_by_color = dict(zip(vals.tolist(), counts.tolist()))

    # Ignore large background/wall colors. Keep small/mid-size objects.
    ignore = set(c for c, n in count_by_color.items() if n > 0.30 * h * w)
    ignore |= set(c for c, n in sorted(count_by_color.items(), key=lambda kv: kv[1], reverse=True)[:2])

    seen = np.zeros((h, w), dtype=bool)
    comps = []
    for y in range(h):
        for x in range(w):
            if seen[y, x]:
                continue
            color = int(arr[y, x])
            if color in ignore:
                seen[y, x] = True
                continue
            q = deque([(x, y)])
            seen[y, x] = True
            pts = []
            while q:
                cx, cy = q.popleft()
                pts.append((cx, cy))
                for nx, ny in ((cx+1, cy), (cx-1, cy), (cx, cy+1), (cx, cy-1)):
                    if 0 <= nx < w and 0 <= ny < h and not seen[ny, nx] and int(arr[ny, nx]) == color:
                        seen[ny, nx] = True
                        q.append((nx, ny))
            area = len(pts)
            if min_area <= area <= max_area:
                xs = [p[0] for p in pts]
                ys = [p[1] for p in pts]
                comps.append({
                    "color": color,
                    "area": int(area),
                    "x0": int(min(xs)), "x1": int(max(xs)),
                    "y0": int(min(ys)), "y1": int(max(ys)),
                    "cx": float(sum(xs) / area),
                    "cy": float(sum(ys) / area),
                    "w": int(max(xs) - min(xs) + 1),
                    "h": int(max(ys) - min(ys) + 1),
                    "touches_edge": bool(min(xs) == 0 or min(ys) == 0 or max(xs) == w-1 or max(ys) == h-1),
                })
    return comps

def rank_candidates(comps, player_pos=None, clicked_memory=None):
    clicked_memory = clicked_memory or {}
    ranked = []
    for c in comps:
        # exclude likely lower UI bars and giant UI/wall fragments
        if c["cy"] > 61 or c["area"] > 180 or c["w"] > 28 or c["h"] > 28:
            continue
        score = 0.0
        score += 1.0 / (1.0 + abs(c["area"] - 12))  # small object prior
        if not c["touches_edge"]:
            score += 0.20
        key = (round(c["cx"]), round(c["cy"]), c["color"])
        score -= 0.08 * clicked_memory.get(key, 0)
        if player_pos is not None:
            d = abs(c["cx"] - player_pos[0]) + abs(c["cy"] - player_pos[1])
            if d < 5:
                score -= 1.5  # avoid targeting the player itself
            else:
                score += 0.01 * min(d, 50)  # novelty away from self
        ranked.append((score, c))
    ranked.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in ranked]

def infer_player_from_diff(prev_frame, cur_frame, old_player_pos=None):
    """Roughly track controllable/moving object from changed pixels."""
    if prev_frame is None or cur_frame is None:
        return old_player_pos
    prev = np.asarray(prev_frame)
    cur = np.asarray(cur_frame)
    if prev.shape != cur.shape:
        return old_player_pos
    d = prev != cur
    ys, xs = np.where(d)
    n = len(xs)
    # Ignore full-screen animation/transition; use object-scale diffs.
    if 2 <= n <= 350:
        cx = float(xs.mean())
        cy = float(ys.mean())
        if old_player_pos is None:
            return (cx, cy)
        return (0.70 * old_player_pos[0] + 0.30 * cx, 0.70 * old_player_pos[1] + 0.30 * cy)
    return old_player_pos

def classify_family(valid_actions):
    has_move = any(a in valid_actions for a in MOVE_ACTIONS)
    has_click = any(a.is_complex() for a in valid_actions)
    if has_move and has_click:
        return "keyboard_click"
    if has_click:
        return "click"
    if has_move:
        return "keyboard"
    return "unknown"

def choose_direction_toward(player_pos, target, valid_actions, blocked_until, step_idx):
    if player_pos is None or target is None:
        return None
    moves = [a for a in MOVE_ACTIONS if a in valid_actions and blocked_until.get(a.name, -1) < step_idx]
    if not moves:
        return None
    px, py = player_pos
    tx, ty = target["cx"], target["cy"]
    current = abs(tx - px) + abs(ty - py)
    scored = []
    for a in moves:
        dx, dy = MOVE_DELTAS[a]
        nx, ny = px + 4 * dx, py + 4 * dy
        nd = abs(tx - nx) + abs(ty - ny)
        scored.append((current - nd, -nd, a))
    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
    return scored[0][2]

def choose_action(obs, env, rng, memory, step_idx):
    frame = get_frame(obs)
    valid = list(getattr(env, "action_space", []))
    if not valid:
        valid = [a for a in GameAction if a is not GameAction.RESET]
    valid = [a for a in valid if a is not GameAction.RESET]

    # Avoid undo by default; it destroyed too many random rollouts.
    no_undo = [a for a in valid if a is not GameAction.ACTION7]
    if no_undo:
        valid = no_undo

    family = classify_family(valid)
    if frame is None:
        simple = [a for a in valid if not a.is_complex()]
        a = rng.choice(simple or valid)
        return a, {}, {"policy": "no_frame_random", "family": family}

    comps = connected_components(frame)
    ranked = rank_candidates(comps, memory.get("player_pos"), memory["clicked_memory"])
    target = ranked[0] if ranked else None
    memory["last_components"] = comps

    click_actions = [a for a in valid if a.is_complex()]
    simple_actions = [a for a in valid if not a.is_complex()]
    moves = [a for a in MOVE_ACTIONS if a in valid and memory["blocked_until"].get(a.name, -1) < step_idx]

    # Policy routing.
    if family == "click":
        if click_actions and ranked:
            idx = memory["click_cursor"] % len(ranked)
            c = ranked[idx]
            memory["click_cursor"] += 1
            data = {"x": int(round(c["cx"])), "y": int(round(c["cy"]))}
            return rng.choice(click_actions), data, {
                "policy": "click_only_component_cycle",
                "family": family,
                "target_color": c["color"], "target_area": c["area"], "target_cx": c["cx"], "target_cy": c["cy"],
            }
        a = rng.choice(click_actions or valid)
        data = random_visible_pixel(frame, rng) if a.is_complex() else {}
        return a, data, {"policy": "click_only_exp001_fallback", "family": family}

    if family == "keyboard":
        move = choose_direction_toward(memory.get("player_pos"), target, valid, memory["blocked_until"], step_idx)
        if move is not None and rng.random() < 0.75:
            return move, {}, {"policy": "keyboard_move_toward_component", "family": family}
        if moves:
            # Prefer moves with lower no-op counts.
            moves_sorted = sorted(moves, key=lambda a: (memory["noop_counts"].get(a.name, 0), rng.random()))
            return moves_sorted[0], {}, {"policy": "keyboard_low_noop_walk", "family": family}
        a = rng.choice(simple_actions or valid)
        return a, {}, {"policy": "keyboard_fallback", "family": family}

    if family == "keyboard_click":
        # In keyboard-click games, movement carried EXP-001; click occasionally on object centers.
        move = choose_direction_toward(memory.get("player_pos"), target, valid, memory["blocked_until"], step_idx)
        if move is not None and rng.random() < 0.65:
            return move, {}, {"policy": "hybrid_move_toward_component", "family": family}
        if click_actions and ranked and rng.random() < 0.30:
            c = ranked[memory["click_cursor"] % len(ranked)]
            memory["click_cursor"] += 1
            data = {"x": int(round(c["cx"])), "y": int(round(c["cy"]))}
            return rng.choice(click_actions), data, {
                "policy": "hybrid_click_component",
                "family": family,
                "target_color": c["color"], "target_area": c["area"], "target_cx": c["cx"], "target_cy": c["cy"],
            }
        if moves:
            moves_sorted = sorted(moves, key=lambda a: (memory["noop_counts"].get(a.name, 0), rng.random()))
            return moves_sorted[0], {}, {"policy": "hybrid_low_noop_walk", "family": family}

    # Unknown/general fallback: EXP-001-like random valid action, with object-center clicks if possible.
    if click_actions and target is not None and rng.random() < 0.5:
        data = {"x": int(round(target["cx"])), "y": int(round(target["cy"]))}
        return rng.choice(click_actions), data, {"policy": "unknown_click_component", "family": family}
    a = rng.choice(simple_actions or valid)
    data = random_visible_pixel(frame, rng) if a.is_complex() else {}
    return a, data, {"policy": "unknown_exp001_fallback", "family": family}

def play(env, game_id):
    rng = rng_for(game_id)
    obs = getattr(env, "_last_response", None)
    memory = {
        "player_pos": None,
        "prev_frame": None,
        "blocked_until": {},
        "noop_counts": defaultdict(int),
        "clicked_memory": defaultdict(int),
        "click_cursor": 0,
        "last_components": [],
    }
    action_counts = defaultdict(int)
    policy_counts = defaultdict(int)
    noop_counts = defaultdict(int)
    family_seen = None
    last_action = None
    last_policy = None

    for step_idx in range(1, MAX_MOVES + 1):
        if obs is None or obs.state == GameState.WIN:
            break
        if obs.state in (GameState.GAME_OVER, GameState.NOT_PLAYED):
            obs = env.step(GameAction.RESET, {})
            last_action = "RESET"
            last_policy = "reset"
            continue

        cur_frame = get_frame(obs)
        if cur_frame is not None:
            memory["player_pos"] = infer_player_from_diff(memory.get("prev_frame"), cur_frame, memory.get("player_pos"))

        action, data, reasoning = choose_action(obs, env, rng, memory, step_idx)
        family_seen = reasoning.get("family", family_seen)

        prev_frame = cur_frame.copy() if cur_frame is not None else None
        prev_hash = frame_hash(prev_frame)
        obs = env.step(action, data=data, reasoning=reasoning)
        next_frame = get_frame(obs)
        next_hash = frame_hash(next_frame)

        last_action = action.name
        last_policy = reasoning.get("policy", "unknown")
        action_counts[action.name] += 1
        policy_counts[last_policy] += 1

        if action.is_complex() and data:
            key = (int(data.get("x", -1)), int(data.get("y", -1)))
            memory["clicked_memory"][key] += 1

        # No-op suppression: if action changed nothing, block it briefly.
        if prev_hash is not None and next_hash == prev_hash:
            memory["noop_counts"][action.name] += 1
            noop_counts[action.name] += 1
            memory["blocked_until"][action.name] = step_idx + min(20, 4 + memory["noop_counts"][action.name])
        else:
            memory["noop_counts"][action.name] = 0

        memory["prev_frame"] = prev_frame

    return {
        "game_id": game_id,
        "moves": int(step_idx),
        "state": obs.state.name if obs else "unknown",
        "levels_completed": int(obs.levels_completed) if obs else 0,
        "family": family_seen,
        "last_action": last_action,
        "last_policy": last_policy,
        "action_counts": dict(action_counts),
        "policy_counts": dict(policy_counts),
        "noop_counts": dict(noop_counts),
    }

arcade = arc_agi.Arcade()
infos = list(arcade.available_environments)
rows = []
details = []

print(EXP_ID, "envs=", len(infos), "MAX_MOVES=", MAX_MOVES, "SEED=", SEED, "USE_PER_GAME_SEED=", USE_PER_GAME_SEED)
for i, info in enumerate(infos, 1):
    row = play(arcade.make(info.game_id), info.game_id)
    details.append(row)
    flat = {k: v for k, v in row.items() if k not in ("action_counts", "policy_counts", "noop_counts")}
    rows.append(flat)
    print(f"[{i:03d}/{len(infos):03d}] {flat}")

pd.DataFrame(rows).to_csv(ART / "exp002_run_results.csv", index=False)
(ART / "exp002_run_details.json").write_text(json.dumps(details, indent=2))

# Aggregate diagnostics.
policy_totals = defaultdict(int)
action_totals = defaultdict(int)
noop_totals = defaultdict(int)
for d in details:
    for k, v in d["policy_counts"].items():
        policy_totals[k] += int(v)
    for k, v in d["action_counts"].items():
        action_totals[k] += int(v)
    for k, v in d["noop_counts"].items():
        noop_totals[k] += int(v)

(ART / "exp002_policy_counts.json").write_text(json.dumps(dict(policy_totals), indent=2, sort_keys=True))
(ART / "exp002_action_counts.json").write_text(json.dumps(dict(action_totals), indent=2, sort_keys=True))
(ART / "exp002_noop_counts.json").write_text(json.dumps(dict(noop_totals), indent=2, sort_keys=True))

summary = {"exp_id": EXP_ID, "max_moves": MAX_MOVES, "seed": SEED, "use_per_game_seed": USE_PER_GAME_SEED}

if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    sc = arcade.get_scorecard()
    summary.update(
        score=float(sc.score),
        total_environments_completed=int(sc.total_environments_completed),
        total_environments=int(sc.total_environments),
        total_levels_completed=int(sc.total_levels_completed),
        total_levels=int(sc.total_levels),
        total_actions=int(sc.total_actions),
    )
    pd.DataFrame([
        {
            "game": e.id,
            "score": float(e.score),
            "levels_completed": int(e.levels_completed),
            "actions": int(e.actions),
            "completed": bool(e.completed),
        }
        for e in sc.environments
    ]).to_csv(ART / "exp002_scorecard_by_environment.csv", index=False)
    pd.DataFrame([
        {
            "tag": t.id,
            "score": float(t.score),
            "levels_completed": int(t.levels_completed),
            "number_of_environments": int(t.number_of_environments),
        }
        for t in (sc.tags_scores or [])
    ]).to_csv(ART / "exp002_scorecard_by_tag.csv", index=False)
    print("Score:", f"{sc.score:.4f}", "Levels:", f"{sc.total_levels_completed}/{sc.total_levels}", "Actions:", sc.total_actions)

(ART / "exp002_scorecard_summary.json").write_text(json.dumps(summary, indent=2))

sp = WORK / "submission.parquet"
if not sp.exists():
    pd.DataFrame([["1_0", "1", True, 1]], columns=["row_id", "game_id", "end_of_game", "score"]).to_parquet(sp, index=False)

manifest = sorted(str(p) for p in ART.glob("*"))
pd.DataFrame({"artifact": manifest}).to_csv(ART / "artifact_manifest.csv", index=False)

print("submission:", sp, sp.exists())
print("artifacts:", ART)
print(json.dumps(summary, indent=2))
summary
